# HTRC New Wave SF — BERTopic Pipeline
Run this notebook inside the HTRC Data Capsule **restricted phase** (when text data is mounted).
Only vectors, metadata, and topic outputs are exported — no raw text leaves the capsule.

## 1. Configuration — edit these paths before running

In [ ]:
from pathlib import Path

# --- Paths ---
# Folder(s) where HTRC plain-text files live inside the capsule.
# Add as many as you have; the code will search each recursively.
TEXT_DIRS = [
    Path('/data/sf_corpus'),        # your SF workset
    Path('/data/fiction_corpus'),   # NovelTM comparison corpus
]

# CSV with at minimum columns: htid, year, corpus
# corpus values should be 'sf' or 'fiction'
METADATA_CSV = Path('/home/dcuser/metadata.csv')

# Where the model was saved in internet phase
MODEL_PATH = Path('/home/dcuser/models/minilm')

# Output directory (everything exported from here)
OUT_DIR = Path('/home/dcuser/outputs')
OUT_DIR.mkdir(exist_ok=True)

# --- Chunking ---
CHUNK_SIZE  = 500   # words per chunk
OVERLAP     = 50    # word overlap between chunks
MIN_WORDS   = 50    # discard chunks shorter than this

# --- Embedding ---
BATCH_SIZE  = 64    # increase if you have lots of RAM; decrease if you get OOM

# --- BERTopic ---
UMAP_COMPONENTS = 5
HDBSCAN_MIN_CLUSTER = 50
N_TOPICS = 'auto'   # or set an integer like 40

## 2. Load metadata

In [ ]:
import pandas as pd

meta_df = pd.read_csv(METADATA_CSV)

# Normalise the htid column so it matches filenames
meta_df['htid'] = meta_df['htid'].str.strip()

# Build a lookup dict: htid -> {year, corpus, title, author}
meta_lookup = meta_df.set_index('htid').to_dict('index')

print(f"Loaded metadata for {len(meta_lookup):,} volumes")
print(meta_df['corpus'].value_counts())
meta_df.head()

## 3. Discover text files and match to metadata

In [ ]:
def htid_from_path(p: Path) -> str:
    """Recover HTID from a file path.
    HTRC capsule files use '+' instead of '/' in the HTID,
    e.g. mdp+39015004000489.txt  ->  mdp.39015004000489
    Also handles plain underscores and dot separators.
    """
    stem = p.stem  # filename without extension
    # Try '+' separator first (most common in capsule)
    if '+' in stem:
        parts = stem.split('+')
        return parts[0] + '.' + parts[1] if len(parts) == 2 else stem
    return stem  # fall back to raw stem


all_files = []
for d in TEXT_DIRS:
    all_files.extend(d.rglob('*.txt'))

# Match files to metadata
matched, skipped = [], []
for f in all_files:
    htid = htid_from_path(f)
    if htid in meta_lookup:
        matched.append((f, htid, meta_lookup[htid]))
    else:
        skipped.append(htid)

print(f"Found {len(all_files):,} files")
print(f"Matched to metadata: {len(matched):,}")
print(f"No metadata (will skip): {len(skipped):,}")
if skipped:
    print("First 5 unmatched:", skipped[:5])

## 4. Chunk all texts

In [ ]:
import json
from tqdm.notebook import tqdm

def chunk_text(text, htid, meta, chunk_size=CHUNK_SIZE, overlap=OVERLAP, min_words=MIN_WORDS):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        word_slice = words[i : i + chunk_size]
        if len(word_slice) < min_words:
            continue
        chunks.append({
            'htid':     htid,
            'chunk_id': f"{htid}__{i}",
            'year':     int(meta.get('year', 0)),
            'corpus':   meta.get('corpus', 'unknown'),
            'title':    meta.get('title', ''),
            'author':   meta.get('author', ''),
            'text':     ' '.join(word_slice),
        })
    return chunks


all_chunks = []
read_errors = []

for fpath, htid, meta in tqdm(matched, desc='Chunking'):
    try:
        text = fpath.read_text(encoding='utf-8', errors='replace')
        all_chunks.extend(chunk_text(text, htid, meta))
    except Exception as e:
        read_errors.append((htid, str(e)))

print(f"\nTotal chunks: {len(all_chunks):,}")
print(f"Read errors:  {len(read_errors)}")

# Quick breakdown
import collections
corpus_counts = collections.Counter(c['corpus'] for c in all_chunks)
print("Chunks by corpus:", dict(corpus_counts))

## 5. Generate embeddings

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(str(MODEL_PATH))

texts = [c['text'] for c in all_chunks]

print(f"Encoding {len(texts):,} chunks with batch_size={BATCH_SIZE}…")
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"Embedding matrix shape: {embeddings.shape}")

# Save immediately — this is the expensive step
np.save(OUT_DIR / 'embeddings.npy', embeddings)
print("Saved embeddings.npy")

## 6. Save chunk metadata (no text — non-consumptive)

In [ ]:
# Strip the text field before saving — only vectors and metadata leave the capsule
meta_out = [{k: v for k, v in c.items() if k != 'text'} for c in all_chunks]

with open(OUT_DIR / 'chunk_metadata.json', 'w') as f:
    json.dump(meta_out, f)

print(f"Saved chunk_metadata.json ({len(meta_out):,} records)")

## 7. Fit BERTopic

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_components=UMAP_COMPONENTS,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER,
    metric='euclidean',
    prediction_data=True,
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=N_TOPICS,
    verbose=True,
)

topics, probs = topic_model.fit_transform(texts, embeddings)

print(f"\nNumber of topics found: {len(set(topics)) - 1} (plus noise topic -1)")

## 8. Inspect topics

In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info[topic_info['Topic'] != -1].head(30).to_string())

In [ ]:
# Show top words for topics you care about — edit topic numbers here
topics_of_interest = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

for t in topics_of_interest:
    words = topic_model.get_topic(t)
    if words:
        top = ', '.join([w for w, _ in words[:10]])
        print(f"Topic {t:3d}: {top}")

## 9. Diachronic topic analysis — topics over time

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless
import matplotlib.pyplot as plt

# Build a dataframe with topic assignments
chunk_df = pd.DataFrame(meta_out)
chunk_df['topic'] = topics

# Filter out noise
chunk_df_clean = chunk_df[chunk_df['topic'] != -1].copy()

# Count topic occurrences per year per corpus
yearly = (
    chunk_df_clean
    .groupby(['year', 'corpus', 'topic'])
    .size()
    .reset_index(name='count')
)

# Normalise: proportion of chunks per year-corpus
totals = chunk_df_clean.groupby(['year', 'corpus']).size().reset_index(name='total')
yearly = yearly.merge(totals, on=['year', 'corpus'])
yearly['proportion'] = yearly['count'] / yearly['total']

yearly.to_csv(OUT_DIR / 'topic_by_year.csv', index=False)
print("Saved topic_by_year.csv")
yearly.head()

In [ ]:
# Plot a single topic over time, split by corpus
# Change PLOT_TOPIC to whichever topic number looks like environmental content
PLOT_TOPIC = 0

fig, ax = plt.subplots(figsize=(10, 4))

for corpus, grp in yearly[yearly['topic'] == PLOT_TOPIC].groupby('corpus'):
    ax.plot(grp['year'], grp['proportion'], marker='o', label=corpus)

ax.set_title(f"Topic {PLOT_TOPIC} prevalence over time")
ax.set_xlabel('Year')
ax.set_ylabel('Proportion of chunks')
ax.legend()
ax.axvline(1962, color='grey', linestyle='--', alpha=0.5, label='Silent Spring')
fig.tight_layout()
fig.savefig(OUT_DIR / f'topic_{PLOT_TOPIC}_over_time.png', dpi=150)
print(f"Saved plot for topic {PLOT_TOPIC}")

## 10. Export everything needed for analysis outside the capsule

In [ ]:
# Save full topic-chunk assignment table
chunk_df.to_csv(OUT_DIR / 'chunk_topic_assignments.csv', index=False)

# Save topic info (keywords, sizes)
topic_info.to_csv(OUT_DIR / 'topic_info.csv', index=False)

# Save top words per topic as JSON
topic_words = {}
for t in topic_info['Topic']:
    if t == -1:
        continue
    words = topic_model.get_topic(t)
    if words:
        topic_words[str(t)] = [{'word': w, 'score': round(s, 5)} for w, s in words]

with open(OUT_DIR / 'topic_words.json', 'w') as f:
    json.dump(topic_words, f, indent=2)

print("Outputs written to", OUT_DIR)
print("Files:")
for p in sorted(OUT_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name:40s}  {size_mb:.1f} MB")

## 11. (Optional) Save BERTopic model
The model itself (without the raw documents) can be saved and potentially re-loaded.

In [ ]:
topic_model.save(
    str(OUT_DIR / 'bertopic_model'),
    serialization='safetensors',
    save_ctfidf=True,
    save_embedding_model=False,  # don't re-bundle the encoder
)
print("BERTopic model saved.")

---
### What to SCP out of the capsule
```
scp -P 16040 -r dcuser@dc6.htrc.indiana.edu:/home/dcuser/outputs/ ./outputs/
```
The `outputs/` folder contains only vectors, counts, and keywords — no raw text, consistent with HTRC non-consumptive use policy.